In [1]:
import torch

from torch_openreml.covariance import (
    DiagonalMatrix,
    ScalarMatrix,
    Sum,
)

torch.set_default_dtype(torch.float64)
torch.set_printoptions(precision=3, sci_mode=False)

op = Sum(
    genetic=ScalarMatrix(3),
    environment=DiagonalMatrix(3),
)
theta = torch.zeros(4)

print("names:", op.param_names)
print(op(theta))
print("shape:", op.shape)

names: ['genetic/sigma^2', 'environment/sigma^2_0', 'environment/sigma^2_1', 'environment/sigma^2_2']
tensor([[2., 0., 0.],
        [0., 2., 0.],
        [0., 0., 2.]])
shape: (3, 3)


In [2]:
op.build_params(theta, out_format="dict")

{'genetic/sigma^2': tensor([1.]),
 'environment/sigma^2_0': tensor([1.]),
 'environment/sigma^2_1': tensor([1.]),
 'environment/sigma^2_2': tensor([1.])}

In [3]:
positional = Sum(ScalarMatrix(2), DiagonalMatrix(2))
print(positional.param_names)

['op_0/sigma^2', 'op_1/sigma^2_0', 'op_1/sigma^2_1']


In [4]:
named = Sum({"left": ScalarMatrix(2), "right": DiagonalMatrix(2)})
print(named.param_names)

['left/sigma^2', 'right/sigma^2_0', 'right/sigma^2_1']


In [5]:
fixed = Sum(
    signal=ScalarMatrix(2),
    baseline=torch.eye(2),
)

print("names:", fixed.param_names)
print(fixed(torch.tensor([0.0])))

names: ['signal/sigma^2']
tensor([[2., 0.],
        [0., 2.]])


In [6]:
mixed = Sum(
    signal=ScalarMatrix(2),
    baseline=torch.eye(2, dtype=torch.float32),
)

print("default build:", mixed(torch.tensor([0.0])).dtype)
print(
    "float32 input:",
    mixed(torch.tensor([0.0], dtype=torch.float32)).dtype,
)

default build: torch.float64
float32 input: torch.float32


In [7]:
from torch_openreml.covariance import BlockDiagonal

nested = Sum(
    inner=BlockDiagonal(
        DiagonalMatrix(2),
        ScalarMatrix(2),
    ),
    extra=ScalarMatrix(4),
)
theta = torch.tensor([0.0, 0.5, 1.0, 0.5])

for path, node, params in nested.walk(theta):
    print(f"{path:16} {type(node).__name__:22} {params}")

/                Sum                    tensor([0.000, 0.500, 1.000, 0.500])
inner            BlockDiagonal          tensor([0.000, 0.500, 1.000])
inner/op_0       DiagonalMatrix         tensor([0.000, 0.500])
inner/op_1       ScalarMatrix           tensor([1.])
extra            ScalarMatrix           tensor([0.500])


In [8]:
results = nested.tree(theta)

print("paths:", list(results))
print(results["inner/op_0"])

paths: ['/', 'inner', 'inner/op_0', 'inner/op_1', 'extra']
tensor([[1.000, 0.000],
        [0.000, 2.718]])


In [9]:
grad, names = op.grad(theta)

print("gradient shape:", grad.shape)
print("gradient names:", names)

gradient shape: torch.Size([4, 3, 3])
gradient names: ['genetic/sigma^2', 'environment/sigma^2_0', 'environment/sigma^2_1', 'environment/sigma^2_2']
